# MLP on Engineered Trajectory Features

Does deep learning beat the Random Forest baseline because raw trajectories carry
more information, or simply because neural nets outperform RF regardless of input?
This notebook answers that by training an MLP on the **same hand-engineered
features** the RF baseline used — so any improvement must come from architecture, not input.

**Input:** `engineered_feature_table.csv` (1 file, 1,030 rows × 47 cols).  
Each row is one pre-aggregated trajectory with features already computed.

**Features used (18 pure trajectory features):**
x/y variance, step sizes, displacement variance, path length, net displacement,
straightness index, turning angle stats (mean/var/median/acf×2),
speed/vector autocorrelations, PSD (power/peak freq/spectral centroid).

**Dropped (leakage / positional):**
- `convex_hull_area_normalized_*` and `radius_of_gyration_normalized_*` → directly encode nucleus area, leak `area_um2`
- `x_nm`, `y_nm` → absolute position
- Nucleus morphology columns (perimeter, circularity, etc.) → not in the RF baseline's trajectory features

---
**Running on Google Colab?**  
1. Upload `engineered_feature_table.csv` to your Google Drive.  
2. Run the *Environment setup* cell — it mounts Drive automatically.  
3. Set `CSV_DRIVE_PATH` in the *Paths & config* cell to point to that file.

## Environment setup

In [ ]:
import sys, os

IN_COLAB = 'google.colab' in sys.modules
print('Running in Colab:', IN_COLAB)

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    print('Drive mounted at /content/drive')

# All packages below are pre-installed in Colab / standard conda envs.
# Uncomment if anything is missing:
# !pip install -q torch numpy pandas scikit-learn

## Paths & config

**Edit `CSV_DRIVE_PATH` if running on Colab.**  
Only one file is needed (`engineered_feature_table.csv`).

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split, KFold, GroupShuffleSplit
from sklearn.metrics import r2_score
from sklearn.ensemble import RandomForestRegressor
import warnings
warnings.filterwarnings('ignore')

# ── Path configuration ─────────────────────────────────────────────────────
import os
# Local: set DATA_ROOT to override the committed modeling data.
_LOCAL_DATA_CANDIDATES = [Path('../data'), Path('data'), Path('trajectory_to_nuclear_features/data')]
LOCAL_BASE = (Path(os.environ['DATA_ROOT']) if 'DATA_ROOT' in os.environ
              else next((p for p in _LOCAL_DATA_CANDIDATES if p.exists()), _LOCAL_DATA_CANDIDATES[0]))
LOCAL_CSV = LOCAL_BASE / 'chr3/engineered_feature_table.csv'

# Colab / Drive — edit to point to your uploaded CSV
CSV_DRIVE_PATH = Path('/content/drive/MyDrive/CS273B/engineered_feature_table.csv')

ENG_CSV = CSV_DRIVE_PATH if IN_COLAB else LOCAL_CSV
print('CSV path:', ENG_CSV)
assert ENG_CSV.exists(), f'File not found: {ENG_CSV}'

# Output dir (results CSV)
OUT_DIR = ENG_CSV.parent

# ── Device ─────────────────────────────────────────────────────────────────
if torch.cuda.is_available():
    DEVICE = 'cuda'
elif torch.backends.mps.is_available():
    DEVICE = 'mps'
else:
    DEVICE = 'cpu'
print(f'PyTorch {torch.__version__} | Device: {DEVICE}')

# ── Targets ────────────────────────────────────────────────────────────────
# nuc_mean_intensity in this CSV == nuc_intensity_mean in run_deep_learning_experiments.py
TARGETS = [
    'area_um2',
    'local_intensity_mean',
    'local_to_nuc_ratio',
    'nuc_mean_intensity',
    'dist_to_membrane_nm',
    'dist_to_centroid_nm',
    'norm_radial_pos',
]
TARGET_DISPLAY = [
    'area_um2',
    'local_intensity_mean',
    'local_to_nuc_ratio',
    'nuc_intensity_mean',
    'dist_to_membrane_nm',
    'dist_to_centroid_nm',
    'norm_radial_pos',
]

# ── Trajectory features (no leakage) ───────────────────────────────────────
TRAJ_FEATURES = [
    'x_variance_nm2',
    'y_variance_nm2',
    'mean_step_size_nm',
    'max_step_size_nm',
    'displacement_variance_nm2',
    'total_path_length_nm',
    'net_displacement_nm',
    'straightness_index',
    'turning_angle_mean',
    'turning_angle_var',
    'turning_angle_median',
    'turning_angle_acf_lag1',
    'turning_angle_acf_lag2',
    'speed_autocorr_lag1',
    'vector_autocorr_lag1',
    'step_size_psd_total_power',
    'step_size_psd_peak_frequency',
    'step_size_psd_spectral_centroid',
]

# ── Hyperparameters ─────────────────────────────────────────────────────────
SEED      = 42
BATCH_SZ  = 32
N_EPOCHS  = 150
LR        = 1e-3
WD        = 1e-4
PATIENCE  = 20
N_FOLDS   = 5

torch.manual_seed(SEED)
np.random.seed(SEED)

## Load and inspect data

In [ ]:
df = pd.read_csv(ENG_CSV)
print(f'Shape: {df.shape}')
print(f'Unique nuclei: {df["nucleus_id"].nunique()}  |  Unique loci: {df["locus_id"].nunique()}')

# Check all expected columns exist
missing_feats = [f for f in TRAJ_FEATURES if f not in df.columns]
missing_tgts  = [t for t in TARGETS if t not in df.columns]
print(f'Missing feature columns: {missing_feats or "none"}')
print(f'Missing target columns:  {missing_tgts or "none"}')

# Filter to G loci only (matches run_deep_learning_experiments.py)
df_g = df[df['locus_id'].str.startswith('G')].copy()
df_g = df_g.dropna(subset=TRAJ_FEATURES, how='all')   # drop rows with no features at all
print(f'\nG-locus rows after filtering: {len(df_g)}')

## Build X and Y arrays

In [ ]:
X_raw   = df_g[TRAJ_FEATURES].values.astype(np.float32)
Y_raw   = df_g[TARGETS].values.astype(np.float32)
nuc_ids = df_g['nucleus_id'].values

print(f'X: {X_raw.shape}  ({len(TRAJ_FEATURES)} features)')
print(f'Y: {Y_raw.shape}')
print(f'Unique nuclei: {len(set(nuc_ids))}')

print('\nPer-feature NaN rate (non-zero only):')
for i, f in enumerate(TRAJ_FEATURES):
    r = np.isnan(X_raw[:, i]).mean()
    if r > 0:
        print(f'  {f}: {r:.1%}')

print('\nTarget NaN counts:')
for i, t in enumerate(TARGETS):
    n = np.isnan(Y_raw[:, i]).sum()
    print(f'  {TARGET_DISPLAY[i]}: {n}')

## Nucleus-level train / test split

In [ ]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=SEED)
tr_idx, te_idx = next(gss.split(X_raw, Y_raw, groups=nuc_ids))

tr_nucs = set(nuc_ids[tr_idx])
te_nucs = set(nuc_ids[te_idx])
overlap  = tr_nucs & te_nucs
print(f'Train: {len(tr_idx)} trajectories, {len(tr_nucs)} unique nuclei')
print(f'Test:  {len(te_idx)} trajectories, {len(te_nucs)} unique nuclei')
print(f'Nucleus overlap: {len(overlap)} ({"✓ none" if not overlap else "✗ OVERLAP"})')

In [ ]:
def norm_X(X_tr, X_te):
    """Per-feature standardisation; NaN → 0 after normalising."""
    mu  = np.nanmean(X_tr, axis=0)
    std = np.nanstd(X_tr,  axis=0) + 1e-8
    Xn_tr = np.where(np.isnan(X_tr), 0.0, (X_tr - mu) / std)
    Xn_te = np.where(np.isnan(X_te), 0.0, (X_te - mu) / std)  # use train stats
    return Xn_tr.astype(np.float32), Xn_te.astype(np.float32)

def norm_Y(Y_tr, Y_te):
    mu  = np.nanmean(Y_tr, axis=0)
    std = np.nanstd(Y_tr,  axis=0) + 1e-8
    return (Y_tr - mu) / std, (Y_te - mu) / std, mu, std

def r2_metric(yt, yp):
    m = ~np.isnan(yt)
    return r2_score(yt[m], yp[m]) if m.sum() >= 2 else np.nan

def pearson_metric(yt, yp):
    m = ~np.isnan(yt)
    return float(np.corrcoef(yt[m], yp[m])[0, 1]) if m.sum() >= 2 else np.nan


X_tr_r = X_raw[tr_idx];  X_te_r = X_raw[te_idx]
Y_tr_r = Y_raw[tr_idx];  Y_te_r = Y_raw[te_idx]

X_tr_n, X_te_n             = norm_X(X_tr_r, X_te_r)
Y_tr_n, Y_te_n, y_mu, y_std = norm_Y(Y_tr_r, Y_te_r)

## MLP architectures

| Variant | Hidden layers         |
|---------|----------------------|
| MLP-2   | 128 → 64             |
| MLP-3   | 256 → 128 → 64       |
| MLP-4   | 512 → 256 → 128 → 64 |

In [ ]:
N_IN = len(TRAJ_FEATURES)

class TabularMLP(nn.Module):
    def __init__(self, n_in, n_out, hidden_dims=(128, 64), drop=0.3):
        super().__init__()
        layers, prev = [], n_in
        for h in hidden_dims:
            layers += [nn.Linear(prev, h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(drop)]
            prev = h
        layers.append(nn.Linear(prev, n_out))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

    def param_count(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


MLP_VARIANTS = [
    ('MLP-2', (128, 64)),
    ('MLP-3', (256, 128, 64)),
    ('MLP-4', (512, 256, 128, 64)),
]

for name, dims in MLP_VARIANTS:
    m = TabularMLP(N_IN, len(TARGETS), hidden_dims=dims)
    print(f'{name}  hidden={dims}  → {m.param_count():,} params')

## Training helpers

In [ ]:
class TabularDataset(Dataset):
    def __init__(self, X, Y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.Y = torch.tensor(Y, dtype=torch.float32)
    def __len__(self): return len(self.X)
    def __getitem__(self, i): return self.X[i], self.Y[i]


def masked_mse(pred, target):
    valid = ~torch.isnan(target)
    loss  = torch.tensor(0.0, device=pred.device)
    for t in range(target.shape[1]):
        m = valid[:, t]
        if m.sum() > 0:
            loss = loss + F.mse_loss(pred[m, t], target[m, t])
    return loss


def fit_mlp(factory, X_tr, Y_tr, X_va, Y_va):
    model = factory().to(DEVICE)
    opt   = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=N_EPOCHS)
    tr_ld = DataLoader(TabularDataset(X_tr, Y_tr), BATCH_SZ, shuffle=True)
    va_ld = DataLoader(TabularDataset(X_va, Y_va), BATCH_SZ)
    best_val, best_state, wait = float('inf'), None, 0
    for _ in range(N_EPOCHS):
        model.train()
        for Xb, Yb in tr_ld:
            opt.zero_grad()
            masked_mse(model(Xb.to(DEVICE)), Yb.to(DEVICE)).backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
        sched.step()
        model.eval()
        val = 0.0
        with torch.no_grad():
            for Xb, Yb in va_ld:
                val += masked_mse(model(Xb.to(DEVICE)), Yb.to(DEVICE)).item()
        val /= len(va_ld)
        if val < best_val - 1e-6:
            best_val   = val
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1
            if wait >= PATIENCE: break
    model.load_state_dict(best_state)
    return model


@torch.no_grad()
def predict_mlp(model, X, Y):
    model.eval()
    out = []
    for Xb, _ in DataLoader(TabularDataset(X, Y), BATCH_SZ):
        out.append(model(Xb.to(DEVICE)).cpu())
    return torch.cat(out).numpy()

## RF baseline on the same nucleus-level split

Re-running RF on the same split gives a fair comparison (the original baseline numbers used a different batch/split).

In [ ]:
print('Training Random Forest baseline ...')

# Impute NaN features with training-set column median for RF
tr_med   = np.nanmedian(X_tr_r, axis=0)
X_tr_imp = np.where(np.isnan(X_tr_r), tr_med, X_tr_r)
X_te_imp = np.where(np.isnan(X_te_r), tr_med, X_te_r)

rf_results = []
for i, (t, td) in enumerate(zip(TARGETS, TARGET_DISPLAY)):
    tr_valid = ~np.isnan(Y_tr_r[:, i])
    te_valid = ~np.isnan(Y_te_r[:, i])
    rf = RandomForestRegressor(n_estimators=200, n_jobs=-1, random_state=SEED)
    rf.fit(X_tr_imp[tr_valid], Y_tr_r[tr_valid, i])
    preds = rf.predict(X_te_imp)
    r2_  = r2_metric(Y_te_r[:, i], preds)
    cor_ = pearson_metric(Y_te_r[:, i], preds)
    print(f'  {td:<24} R²={r2_:.3f}  corr={cor_:.3f}')
    rf_results.append({'target': td, 'model': 'RF-engineered',
                       'cv_r2': np.nan, 'test_r2': r2_, 'test_corr': cor_})

## MLP sweep

In [ ]:
kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
fi_final, vi_final = train_test_split(
    np.arange(len(X_tr_n)), test_size=0.1, random_state=SEED)

mlp_results = []

for label, dims in MLP_VARIANTS:
    factory   = lambda d=dims: TabularMLP(N_IN, len(TARGETS), hidden_dims=d)
    param_cnt = factory().param_count()
    print(f'\n{"─"*60}')
    print(f'{label}  hidden={dims}  ({param_cnt:,} params)')
    print(f'{"─"*60}')

    cv_preds = np.full_like(Y_tr_n, np.nan)
    for fold, (fi, vi) in enumerate(kf.split(X_tr_n)):
        print(f'  Fold {fold+1}/{N_FOLDS}...', end=' ', flush=True)
        m = fit_mlp(factory, X_tr_n[fi], Y_tr_n[fi], X_tr_n[vi], Y_tr_n[vi])
        cv_preds[vi] = predict_mlp(m, X_tr_n[vi], Y_tr_n[vi])
        print('done')

    print('  Final model...', end=' ', flush=True)
    final = fit_mlp(factory,
                    X_tr_n[fi_final], Y_tr_n[fi_final],
                    X_tr_n[vi_final], Y_tr_n[vi_final])
    print('done')

    te_preds = predict_mlp(final, X_te_n, Y_te_n) * y_std + y_mu

    for i, (t, td) in enumerate(zip(TARGETS, TARGET_DISPLAY)):
        cv_r2_ = r2_metric(Y_tr_n[:, i],  cv_preds[:, i])
        te_r2_ = r2_metric(Y_te_r[:, i],  te_preds[:, i])
        te_cor = pearson_metric(Y_te_r[:, i], te_preds[:, i])
        print(f'  {td:<24} CV R²={cv_r2_:>6.3f}  Test R²={te_r2_:>6.3f}  corr={te_cor:>6.3f}')
        mlp_results.append({
            'target': td, 'model': label, 'params': param_cnt,
            'cv_r2': cv_r2_, 'test_r2': te_r2_, 'test_corr': te_cor,
        })

print('\nMLP sweep complete.')

## Full comparison table

In [ ]:
# Original RF baseline numbers (batch 1, random split)
rf_baseline = {
    'area_um2':            0.39,
    'local_intensity_mean':0.26,
    'local_to_nuc_ratio':  None,
    'nuc_intensity_mean':  0.25,
    'dist_to_membrane_nm': None,
    'dist_to_centroid_nm': None,
    'norm_radial_pos':     None,
}

# Raw-trajectory DL results from run_deep_learning_experiments.py (CNN / LSTM)
raw_dl_cnn  = {'area_um2': 0.150, 'local_intensity_mean': 0.640,
               'local_to_nuc_ratio': 0.002, 'nuc_intensity_mean': 0.621,
               'dist_to_membrane_nm': 0.065, 'dist_to_centroid_nm': None, 'norm_radial_pos': -0.005}
raw_dl_lstm = {'area_um2': 0.257, 'local_intensity_mean': 0.621,
               'local_to_nuc_ratio': -0.097, 'nuc_intensity_mean': 0.621,
               'dist_to_membrane_nm': -0.082, 'dist_to_centroid_nm': None, 'norm_radial_pos': -0.169}

all_results = rf_results + mlp_results
res_df      = pd.DataFrame(all_results)

SEP = '=' * 110
print(SEP)
print('COMPARISON — Test R²')
print('RF-base: original batch1 | RF-eng: same nucleus split | MLP: same split | CNN/LSTM: raw steps')
print(SEP)

header = (f'{"Target":<24} {"RF-base":>9} {"RF-eng":>9}'
          + ''.join(f'  {lbl:>9}' for lbl, _ in MLP_VARIANTS)
          + f'  {"CNN-raw":>9}  {"LSTM-raw":>9}')
print(header)
print('-' * 110)

for td in TARGET_DISPLAY:
    xrf = rf_baseline.get(td)
    xrf_s = f'{xrf:.3f}' if xrf is not None else '   (new)'

    rf_row = res_df[(res_df.target == td) & (res_df.model == 'RF-engineered')]
    rf_s   = f'{rf_row.iloc[0].test_r2:.3f}' if len(rf_row) else '     —'

    mlp_cols = []
    for lbl, _ in MLP_VARIANTS:
        row = res_df[(res_df.target == td) & (res_df.model == lbl)]
        mlp_cols.append(f'{row.iloc[0].test_r2:.3f}' if len(row) else '     —')

    cnn_v  = raw_dl_cnn.get(td)
    lstm_v = raw_dl_lstm.get(td)
    cnn_s  = f'{cnn_v:.3f}'  if cnn_v  is not None else '     —'
    lstm_s = f'{lstm_v:.3f}' if lstm_v is not None else '     —'

    row_str = (f'{td:<24} {xrf_s:>9} {rf_s:>9}'
               + ''.join(f'  {v:>9}' for v in mlp_cols)
               + f'  {cnn_s:>9}  {lstm_s:>9}')
    print(row_str)

print(SEP)

In [ ]:
out_path = OUT_DIR / 'mlp_engineered_results.csv'
res_df.to_csv(out_path, index=False)
print(f'Saved to {out_path}')

if IN_COLAB:
    res_df.to_csv('mlp_engineered_results.csv', index=False)
    from google.colab import files
    files.download('mlp_engineered_results.csv')

## Interpretation guide

| Result pattern | Conclusion |
|----------------|------------|
| MLP-eng ≈ RF-eng, raw DL >> both | Raw representation genuinely better than hand-engineered features |
| MLP-eng >> RF-eng, raw DL ≈ MLP-eng | Neural nets beat RF regardless; engineered features are sufficient |
| MLP-eng >> RF-eng, raw DL >> MLP-eng | Both representation AND architecture matter |
| All ≈ similar | Task too hard or dataset too small regardless of approach |